# Logistic regression without using the conjunctive rules feature

## Imports

In [1]:
import mlflow
import pandas as pd
from sklearn.model_selection import train_test_split

from heart_failure.modeling.train import optuna_search_lr
from heart_failure.config.modeling import CV_SPLITS, LR_N_TRIALS, LR_SCORING, F_BETA_THRESHOLD
from heart_failure.config.config import INTERIM_DATA_DIR, HEART_DISEASE, SEX, PROCESSED_DATA_DIR
from heart_failure.config.features import TEST_SIZE, RANDOM_STATE, VAL_SIZE
from heart_failure.reports import print_best_params
from heart_failure.modeling.evaluate import (
    get_metrics, log_metrics, optuna_cv_results_to_df, find_best_threshold
)
from heart_failure.modeling.predict import predict_by_threshold

2026-06-16 18:04:41.387 | INFO     | heart_failure.config.config:<module>:11 - PROJ_ROOT path is: D:\heart_failure


In [2]:
MODEL_NAME = "logistic_regression_without_conj_feature"
mlflow.set_experiment(f"heart_failure")

<Experiment: artifact_location='file:D:/heart_failure/notebooks/mlruns/1', creation_time=1781534399229, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781534399229, lifecycle_stage='active', name='heart_failure', tags={}, trace_location=None, workspace='default'>

In [3]:
df = pd.read_csv(INTERIM_DATA_DIR / "heart.csv")
rem_columns = pd.read_csv(PROCESSED_DATA_DIR / "remaining_cols.csv")
rem_columns = rem_columns.squeeze().to_list()
rem_columns.remove("remainder__conjunctive_rules")

y = df[HEART_DISEASE]
X = df.drop(columns=[HEART_DISEASE])

stratify = df[[SEX, HEART_DISEASE]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify
)
val_stratify = [X_train[SEX], y_train]
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train
)

## Training

In [ ]:
study, model = optuna_search_lr(
    X_train,
    y_train,
    n_trials=LR_N_TRIALS,
    n_splits=CV_SPLITS,
    scoring=LR_SCORING,
    random_state=RANDOM_STATE,
)

In [5]:
print_best_params(study)

Best parameters: {'model__solver': 'liblinear', 'model__C': 2.6538576339268767, 'model__max_iter': 147, 'model__tol': 0.00023986954226525172, 'model__l1_ratio': 1}
Best score: 0.94151


In [6]:
lr = model.named_steps["model"]
weights = pd.Series(lr.coef_.ravel(), index=rem_columns)
weights.sort_values(key=abs, ascending=False)

target_encoder__ST_Slope          1.108522
target_encoder__ChestPainType     0.856822
binary_encoder__Sex               0.626496
binary_encoder__FastingBS         0.550447
binary_encoder__ExerciseAngina    0.483733
remainder__Oldpeak                0.261094
binarizer__Cholesterol            0.260849
remainder__MaxHR_z               -0.211032
binarizer__Age                    0.147107
binarizer__MaxHR                  0.000000
dtype: float64

The cv score remained virtually unchanged (compared to the model with the existing feature). However, oldpeak also has a relatively low weight. This may also be due to its relationship with ST_Slope (was found in EDA).

### Threshold searching

In [7]:
y_val_proba = model.predict_proba(X_val)[:, 1]
best_threshold = find_best_threshold(y_val, y_val_proba, F_BETA_THRESHOLD)
best_threshold

np.float64(0.46)

## Scores

In [8]:
print("Train metrics")
y_train_pred = predict_by_threshold(model, X_train, best_threshold)
y_train_proba = model.predict_proba(X_train)[:, 1]
train_metrics = get_metrics(y_train, y_train_pred, y_train_proba)
train_metrics

Train metrics


precision    0.859008
recall       0.903846
f2_score     0.894508
pr_auc       0.942868
dtype: float64

The metrics are even slightly worse than those of a baseline. That is, the feature is useful. Most likely, there are some nonlinear dependencies. For this, I want to build a decision tree next.

In [9]:
with mlflow.start_run(run_name=MODEL_NAME) as run:
    mlflow.log_param("model", MODEL_NAME)
    mlflow.log_param("test_size", TEST_SIZE)
    mlflow.log_param("cv_splits", CV_SPLITS)
    mlflow.log_param("optuna_n_trials", LR_N_TRIALS)
    mlflow.log_param("remaining_columns", rem_columns)

    log_metrics(train_metrics, "train_")